### Import Stations 

Exclude stations with more than 10% missing values.

In [ ]:
# Import the tuecycle package
from tuecycle.src.tuecycle import DataManager, get_station
from tuecycle.src.tuecycle.config import list_stations_with_weather
from tuecycle.src.tuecycle.plots import get_plot, list_plots
import pandas as pd



START_DATE = (2022, 12, 1)
END_DATE = (2025, 11, 30)

# Stations with more than 10% missing values
EXCLUDED_STATIONS = [
    "freiburg_dreisam", 
    "heidelberg_berliner",
    "heidelberg_ernst_walz", 
    "heidelberg_gaisberg", 
    "heidelberg_kurfuersten", 
    "heidelberg_liebermann",
    "heidelberg_mannheimer",
    "heidelberg_schlierbacher", 
    "heidelberg_ziegelhaeuser", 
    "heilbronn_sued", 
    "mannheim_jungbusch", 
    "offenburg_haupt",
    ] 

# Stations to load
STATIONS = [station for station in list_stations_with_weather() if station not in EXCLUDED_STATIONS]


# Initialize the DataManager
dm = DataManager(start_date=START_DATE,
                 end_date=END_DATE)
# Load all comparison stations at once using the configuration
dfs = dm.get_multiple(STATIONS)
print(f"Loaded {len(STATIONS)} stations")

for alias in STATIONS:
    print(alias)

# excluded = dm.get_multiple(EXCLUDED_STATIONS)
# Now dfs contains cleaned dataframes for all stations
# plot time series for all stations
df = dfs["ulm_lupfer"]
station = get_station(alias)
fig = get_plot("time_series")(df, title=station.display_name)
fig.show()

# excluded_dfs = dm.get_multiple(EXCLUDED_STATIONS)
# print(f"Loaded {len(EXCLUDED_STATIONS)} excluded stations")
# # print amount of missing values for excluded stations
# for alias in EXCLUDED_STATIONS:
#     df = excluded_dfs[alias]
#     station = get_station(alias)
#     missing_percentage = df['bike'].isna().mean() * 100
#     print(f"Station: {station.display_name}, Missing Values: {missing_percentage:.2f}%")

Loaded 49 stations
freiburg_eschholz
freiburg_gueterbahn
freiburg_wiwili
heidelberg_eppelheimer
heidelberg_ploeck
heidelberg_rohrbacher
heidelberg_theodor_heuss
heilbronn_neckarufer
heilbronn_nord
karlsruhe_erbprinzen
kirchheim_barometer
konstanz_herose
loerrach_berliner
loerrach_friedhof
ludwigsburg_alleen
ludwigsburg_favorite
ludwigsburg_neckarbruecke
mannheim_feudenheimstr_aus
mannheim_feudenheimstr_ein
mannheim_konrad_adenauer
mannheim_kurpfalz
mannheim_lindenhof
mannheim_luzenberg
mannheim_renz
mannheim_schlosspark
mannheim_schwetzinger
ravensburg_bahnhof
ravensburg_eishalle
ravensburg_meer_ab
ravensburg_meer_auf
stuttgart_boeblinger
stuttgart_insel
stuttgart_kirchheimer
stuttgart_koenig_karls
stuttgart_kraeherwald
stuttgart_kremmler
stuttgart_lautenschlager
stuttgart_neckartal
stuttgart_samara
stuttgart_solitude
stuttgart_stuttgarter
stuttgart_taubenheim
stuttgart_waiblinger
stuttgart_waldburg
stuttgart_tuebinger
tuebingen_hirschau
tuebingen_tunnel
tuebingen_steinlach
ulm_lupfer


Remove all visual outliers

In [21]:
VISUAL_OUTLIERS = [
    ("freiburg_eschholz", "2025-04-06 11:00:00"),
    
    ("freiburg_wiwili", "2025-01-30 17:00:00"),
    ("freiburg_wiwili", "2025-01-30 18:00:00"),
    ("freiburg_wiwili", "2025-01-30 19:00:00"), 
    
    ("heidelberg_ploeck", "2025-04-27 08:00:00"),
    
    ("karlsruhe_erbprinzen", "2023-03-28 08:00:00"),
    ("karlsruhe_erbprinzen", "2024-06-07 19:00:00"),
    
    ("kirchheim_barometer", "2022-12-19 19:00:00"),
    
    ("loerrach_berliner", "2023-02-27 17:00:00"),
    
    ("loerrach_friedhof", "2023-05-24 21:00:00"),
    
    ("mannheim_feudenheimstr_aus", "2024-01-27 17:00:00"),
    
    ("mannheim_feudenheimstr_ein", "2024-01-27 15:00:00"),
    
    ("mannheim_schlosspark", "2025-06-27 09:00:00"),
    ("mannheim_schlosspark", "2024-06-12 09:00:00"),
    
    ("ravensburg_meer_auf", "2025-11-12 22:00:00"),
    ("ravensburg_meer_auf", "2023-08-20 23:00:00"),
    ("ravensburg_meer_auf", "2024-03-02 17:00:00"),
    
    ("stuttgart_kremmler", "2024-09-15 10:00:00"),
    ("stuttgart_kremmler", "2024-09-15 11:00:00"),
    ("stuttgart_kremmler", "2024-09-15 12:00:00"),
    ("stuttgart_kremmler", "2024-09-15 13:00:00"),
    
    ("stuttgart_solitude", "2023-10-08 15:00:00"),
    ("stuttgart_solitude", "2025-07-04 21:00:00"),
    
    ("stuttgart_waiblinger", "2023-12-21 09:00:00"),
    ("stuttgart_waiblinger", "2023-12-21 10:00:00"),
    ("stuttgart_waiblinger", "2025-09-21 16:00:00"),
    
    ("tuebingen_steinlach", "2025-03-19 14:00:00"),
]

def remove_outliers(dfs, outliers):
    """Remove outliers from the dataframes."""
    for alias, timestamp in outliers:
        df = dfs[alias]
        df.loc[df["datetime"] == timestamp, 'bike'] = None
    return dfs

dfs = remove_outliers(dfs, VISUAL_OUTLIERS)
print("Visual outliers removed.")


Visual outliers removed.


Remove obviously faulty days (more than ten 0 counts)

In [22]:
# Remove days that have more than ten 0 counts
def remove_faulty_days(dfs, max_zero_counts=10):
    """Remove days with more than max_zero_counts zero bike counts."""
    for alias, df in dfs.items():
        # Count zero bike counts per day
        df['date'] = df['datetime'].dt.date
        zero_counts_per_day = df.groupby('date')['bike'].apply(lambda x: (x == 0).sum())
        
        # Identify faulty days
        faulty_days = zero_counts_per_day[zero_counts_per_day > max_zero_counts].index
        
        # Print faulty days for debugging
        if faulty_days.any():
            print(f"Station {alias} - Faulty days removed: {list(faulty_days)}")
        
        # Remove faulty days
        for day in faulty_days:
            df.loc[df['date'] == day, 'bike'] = None
        
        # Drop the temporary 'date' column
        df.drop(columns=['date'], inplace=True)
    return dfs

dfs = remove_faulty_days(dfs, max_zero_counts=10)
print("Faulty days removed.")

# handily remove Heilbronn Neckarufer outlier period (2025-10-26 00:00:00 - 2025-10-30 00:00:00)
def remove_heilbronn_neckarufer_outlier(dfs):
    alias = "heilbronn_neckarufer"
    df = dfs[alias]
    start_outlier = pd.to_datetime("2025-10-26 00:00:00")
    end_outlier = pd.to_datetime("2025-10-30 00:00:00")
    df.loc[(df["datetime"] >= start_outlier) & (df["datetime"] <= end_outlier), 'bike'] = None
    print(f"Station {alias} - Heilbronn Neckarufer outlier period removed.")
    return dfs
dfs = remove_heilbronn_neckarufer_outlier(dfs)

# Now dfs contains cleaned dataframes for all stations
# # plot time series for all stations
# for alias in STATIONS:
#     df = dfs[alias]
#     station = get_station(alias)
#     fig = get_plot("time_series")(df, title=station.display_name)
#     fig.show()

Station heidelberg_rohrbacher - Faulty days removed: [datetime.date(2024, 5, 6), datetime.date(2024, 5, 7), datetime.date(2024, 5, 8), datetime.date(2024, 5, 9), datetime.date(2024, 5, 10), datetime.date(2024, 5, 11), datetime.date(2024, 5, 12), datetime.date(2024, 5, 13), datetime.date(2024, 5, 14), datetime.date(2024, 5, 15), datetime.date(2024, 5, 16), datetime.date(2024, 5, 17), datetime.date(2024, 5, 18), datetime.date(2024, 5, 19), datetime.date(2024, 5, 20), datetime.date(2024, 5, 21), datetime.date(2024, 5, 22), datetime.date(2024, 5, 23), datetime.date(2024, 5, 24), datetime.date(2024, 5, 25), datetime.date(2024, 5, 26), datetime.date(2024, 5, 27), datetime.date(2024, 5, 28), datetime.date(2024, 5, 29), datetime.date(2024, 5, 30), datetime.date(2024, 5, 31), datetime.date(2024, 6, 1), datetime.date(2024, 6, 2), datetime.date(2024, 6, 3), datetime.date(2024, 6, 4), datetime.date(2024, 6, 5), datetime.date(2024, 6, 6), datetime.date(2024, 6, 7), datetime.date(2024, 6, 8), dateti

Check for more than 10% missing values.

In [23]:
# now check for more than 10% missing values
for alias in STATIONS:
    df = dfs[alias]
    total_count = len(df)
    missing_count = df['bike'].isna().sum()
    missing_percentage = (missing_count / total_count) * 100
    if missing_percentage > 10:
        # add station to EXCLUDED_STATIONS
        EXCLUDED_STATIONS.append(alias)
        # and remove from dfs
        dfs.pop(alias)
        STATIONS.remove(alias)
        print(f"Station {alias} has {missing_percentage:.2f}% missing values and is excluded.")

Station heidelberg_rohrbacher has 10.68% missing values and is excluded.
Station mannheim_lindenhof has 17.57% missing values and is excluded.
Station tuebingen_hirschau has 16.12% missing values and is excluded.


In [38]:
counters = [get_station(station).counter_name for station in STATIONS]
excluded_counters = [get_station(station).counter_name for station in EXCLUDED_STATIONS]
print(f"Number of included stations: {len(STATIONS)}")
print(f"Number of excluded stations: {len(EXCLUDED_STATIONS)}")
for counter in counters:
    print(f"counter_sites: {counter}")
for counter in excluded_counters:
    print(f"excluded_counter_sites: {counter}")
    
import csv
counter_sites = set()
excluded_sites = set()
with open('eco-counter/all_cities/2025/03.csv', 'r') as file:
    reader = csv.reader(file)
    header = next(reader)  # Read the header
    for row in reader:
        if row[3] in counters:
            counter_sites.add((row[1], row[3], row[6], row[7]))  # city in second, counter_site is in the fourth column
        if row[3] in excluded_counters:
            excluded_sites.add((row[1], row[3], row[6], row[7]))  # city in second, counter_site is in the fourth column
with open('eco-counter/all_cities/2023/12.csv', 'r') as file:
    reader = csv.reader(file)
    header = next(reader)  # Read the header
    for row in reader:
        if row[3] in counters:
            counter_sites.add((row[1], row[3], row[6], row[7]))  # city in second, counter_site is in the fourth column
        if row[3] in excluded_counters:
            excluded_sites.add((row[1], row[3], row[6], row[7]))  # city in second, counter_site is in the fourth column
        
# use plotly to plot the counter sites on a map
import plotly.express as px
import pandas as pd 
df = pd.DataFrame(list(counter_sites), columns=['city', 'counter_site', 'longitude', 'latitude'])
df_excluded = pd.DataFrame(list(excluded_sites), columns=['city', 'counter_site', 'longitude', 'latitude'])
df['longitude'] = pd.to_numeric(df['longitude'])
df['latitude'] = pd.to_numeric(df['latitude'])      
df_excluded['longitude'] = pd.to_numeric(df_excluded['longitude'])
df_excluded['latitude'] = pd.to_numeric(df_excluded['latitude'])

fig = px.scatter_map(df, lat="latitude", lon="longitude", hover_name="counter_site", hover_data=["city"],
                        color_discrete_sequence=["fuchsia"], zoom=5, height=600)
fig_excluded = px.scatter_map(df_excluded, lat="latitude", lon="longitude", hover_name="counter_site", hover_data=["city"],
                        color_discrete_sequence=["blue"], zoom=5, height=600)
fig.add_trace(fig_excluded.data[0])
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()  

Number of included stations: 46
Number of excluded stations: 15
counter_sites: FR3 Eschholzstr. / Egonstr. einzeln
counter_sites: FR2 Güterbahn / Ferd.-Weiß-Str.
counter_sites: Wiwilibrücke
counter_sites: Eppelheimer Str. Querschnitt
counter_sites: Plöck
counter_sites: Thedor-Heuss-Brücke Querschnitt
counter_sites: Neckarufer
counter_sites: Route Nord
counter_sites: Erbprinzenstraße
counter_sites: Barometer Kirchheim u. Teck
counter_sites: Herosepark
counter_sites: Berliner Platz
counter_sites: Untere Hartmattenstraße / Hauptfriedhof
counter_sites: Alleenstraße
counter_sites: Marbacher Straße - Favoritepark
counter_sites: Marbacher Straße - Neckarbrücke
counter_sites: Feudenheimstr. stadtauswärts
counter_sites: Feudenheimerstr. stadteinwärts
counter_sites: Konrad-Adenauer-Brücke
counter_sites: Kurpfalzbrücke
counter_sites: Luzenbergstr.
counter_sites: Renzstraße
counter_sites: Schlosspark Lindenhof (Richtung Jugendherberge)
counter_sites: Neckarauer Übergang -Schwetzinger Str.
counter_

In [25]:
# public holidays in baden-wuerttemberg
public_holidays = [
    "2022-12-25",
    "2022-12-26",
    "2023-01-01",
    "2023-01-06",
    "2023-04-07",
    "2023-04-10",
    "2023-05-01",
    "2023-05-18",
    "2023-05-29",
    "2023-06-08",
    "2023-10-03",
    "2023-11-01",
    "2023-12-25",
    "2023-12-26",
    "2024-01-01",
    "2024-01-06",
    "2024-03-29",
    "2024-04-01",
    "2024-05-01",
    "2024-05-09",
    "2024-05-20",
    "2024-05-30",
    "2024-10-03",
    "2024-11-01",
    "2024-12-25",
    "2024-12-26",
    "2025-01-01",
    "2025-01-06",
    "2025-04-18",
    "2025-04-21",
    "2025-05-01",
    "2025-05-29",
    "2025-06-09",
    "2025-06-19",
    "2025-10-03",
    "2025-11-01"
]

# school holidays in baden-wuerttemberg (start date, end date) (inclusive)
school_holidays = [ 
    ("2022-12-21", "2023-01-08"),
    ("2023-04-06", "2023-04-16"),
    ("2023-05-27", "2023-06-11"),
    ("2023-07-27", "2023-09-10"),
    ("2023-10-30", "2023-11-05"),
    ("2023-12-23", "2024-01-07"),
    ("2024-03-23", "2024-04-07"),
    ("2024-05-18", "2024-06-02"),
    ("2024-07-25", "2024-09-08"),
    ("2024-10-26", "2024-11-03"),
    ("2024-12-21", "2025-01-05"),
    ("2025-04-12", "2025-04-27"),
    ("2025-06-07", "2025-06-22"),
    ("2025-07-31", "2025-09-14"),
    ("2025-10-25", "2025-11-02")]

# semester holidays for biggest university in each city
lecture_free_periods = {
    "freiburg": [("2022-12-23", "2023-01-08"),
                 ("2023-02-11", "2023-04-16"), 
                 ("2023-05-27", "2023-06-04"), 
                 ("2023-07-22", "2023-10-15"),
                 ("2023-12-23", "2024-01-07"), 
                 ("2024-02-10", "2024-04-14"),
                 ("2024-05-18", "2024-05-26"), 
                 ("2024-07-27", "2024-10-14"),
                 ("2024-12-21", "2025-01-06"),
                 ("2025-02-08", "2025-04-13"),
                 ("2025-06-07", "2025-06-15"),
                 ("2025-07-19", "2025-10-12")],
    "heidelberg": [("2022-12-22", "2023-01-08"),
                   ("2023-02-18", "2023-04-16"),
                   ("2023-07-29", "2023-10-15"),
                   ("2023-12-21", "2024-01-07"),
                   ("2024-02-10", "2024-04-14"),
                   ("2024-07-27", "2024-10-13"),
                   ("2024-12-21", "2025-01-06"),
                   ("2025-02-08", "2025-04-13"),
                   ("2025-07-26", "2025-10-12")],
    "heilbronn": [("2022-12-24", "2023-01-08"),
                  ("2023-01-21", "2023-03-12"),
                  ("2023-05-27", "2023-06-04"),
                  ("2023-07-01", "2023-09-24"), # end estimated
                  ("2023-12-23", "2024-01-07"), # christmas break estimated
                  ("2024-01-20", "2024-03-10"), # start estimated
                  ("2024-05-18", "2024-05-24"),
                  ("2024-06-29", "2024-09-22"),
                  ("2024-12-21", "2025-01-06"),
                  ("2025-01-18", "2025-03-09"),
                  ("2025-06-07", "2025-06-15"),
                  ("2025-06-28", "2025-09-21")],
    "karlsruhe": [("2022-12-24", "2023-01-08"),
                  ("2023-02-18", "2023-04-16"),
                  ("2023-05-27", "2023-06-04"),
                  ("2023-07-29", "2023-10-22"),
                  ("2023-12-23", "2024-01-07"),
                  ("2024-02-17", "2024-04-14"),
                  ("2024-05-18", "2024-05-26"),
                  ("2024-07-27", "2024-10-20"),
                  ("2024-12-23", "2025-01-06"), # educated guess
                  ("2025-02-15", "2025-04-21"),
                  ("2025-06-07", "2025-06-15"),
                  ("2025-08-02", "2025-10-26")],
    "kirchheim": [], # no university
    "konstanz": [("2022-12-24", "2023-01-08"),
                 ("2023-02-11", "2023-04-10"),
                 ("2023-06-03", "2023-06-11"),
                 ("2023-07-22", "2023-10-22"),
                 ("2023-12-23", "2024-01-07"),
                 ("2024-02-10", "2024-04-07"),
                 ("2024-05-25", "2024-06-02"),
                 ("2024-07-20", "2024-10-20"),
                 ("2024-12-21", "2025-01-06"),
                 ("2025-02-08", "2025-04-06"),
                 ("2025-06-14", "2025-06-22"),
                 ("2025-07-19", "2025-10-19")],
    "loerrach": [], # only dual study programs - no semester breaks
    "ludwigsburg": [("2022-12-24", "2023-01-08"),
                    ("2023-02-04", "2023-04-10"),
                    ("2023-05-27", "2023-06-04"),
                    ("2023-07-22", "2023-10-15"),
                    ("2023-12-23", "2024-01-07"),
                    ("2024-02-03", "2024-04-07"),
                    ("2024-05-18", "2024-05-26"),
                    ("2024-07-20", "2024-10-13"),
                    ("2024-12-21", "2025-01-06"),
                    ("2025-02-01", "2025-04-06"),
                    ("2025-06-07", "2025-06-15"),
                    ("2025-07-26", "2025-10-19")],
    "mannheim": [("2022-12-17", "2023-01-08"),
                 ("2023-01-28", "2023-03-19"),
                 ("2023-07-15", "2023-09-21"),
                 ("2023-12-23", "2024-01-07"),
                 ("2024-01-20", "2024-03-17"),
                 ("2024-07-13", "2024-09-22"),
                 ("2024-12-21", "2025-01-06"),
                 ("2025-01-18", "2025-03-16"), 
                 ("2025-07-12", "2025-09-21")],
    "ravensburg": [("2022-12-24", "2023-01-08"),
                   ("2023-01-28", "2023-03-12"),
                   ("2023-04-06", "2023-04-11"),
                   ("2023-05-27", "2023-06-04"),
                   ("2023-07-01", "2023-10-03"),
                   ("2023-12-23", "2024-01-07"),
                   ("2024-01-27", "2024-03-10"),
                   ("2024-03-28", "2024-04-02"),
                   ("2024-05-18", "2024-05-26"),
                   ("2024-06-29", "2024-10-06"),
                   ("2024-12-21", "2025-01-06"),
                   ("2025-02-01", "2025-03-16"),
                   ("2025-04-17", "2025-04-22"),
                   ("2025-06-07", "2025-06-15"),
                   ("2025-07-05", "2025-10-05")],
    "stuttgart": [("2022-12-24", "2023-01-08"), # educated guess
                  ("2023-02-04", "2023-04-10"), # start estimated
                  ("2023-05-27", "2023-06-04"),
                  ("2023-07-22", "2023-10-15"),
                  ("2023-12-23", "2024-01-07"), 
                  ("2024-02-10", "2024-04-07"),
                  ("2024-05-18", "2024-05-26"),
                  ("2024-07-20", "2024-10-13"),
                  ("2024-12-21", "2025-01-06"),
                  ("2025-02-08", "2025-04-06"),
                  ("2025-06-07", "2025-06-15"),
                  ("2025-07-19", "2025-10-12")],
    "tuebingen": [("2022-12-24", "2023-01-08"),
                  ("2023-02-11", "2023-04-16"),
                  ("2023-05-27", "2023-06-04"),
                  ("2023-07-29", "2023-10-15"),
                  ("2023-12-23", "2024-01-07"),
                  ("2024-02-10", "2024-04-14"),
                  ("2024-05-18", "2024-05-26"),
                  ("2024-07-27", "2024-10-13"),
                  ("2024-12-21", "2025-01-06"),
                  ("2025-02-08", "2025-04-13"),
                  ("2025-06-07", "2025-06-15"),
                  ("2025-07-26", "2025-10-12")],
    "ulm": [("2022-12-24", "2023-01-08"),
            ("2023-02-18", "2023-04-16"),
            ("2023-07-22", "2023-10-15"),
            ("2023-12-23", "2024-01-07"),
            ("2024-02-17", "2024-04-14"),
            ("2024-07-20", "2024-10-13"),
            ("2024-12-24", "2025-01-06"),
            ("2025-02-15", "2025-04-21"),
            ("2025-07-26", "2025-10-12")]
                    
}

for station in STATIONS:
    print(f"Station: {station}")

Station: freiburg_eschholz
Station: freiburg_gueterbahn
Station: freiburg_wiwili
Station: heidelberg_eppelheimer
Station: heidelberg_ploeck
Station: heidelberg_theodor_heuss
Station: heilbronn_neckarufer
Station: heilbronn_nord
Station: karlsruhe_erbprinzen
Station: kirchheim_barometer
Station: konstanz_herose
Station: loerrach_berliner
Station: loerrach_friedhof
Station: ludwigsburg_alleen
Station: ludwigsburg_favorite
Station: ludwigsburg_neckarbruecke
Station: mannheim_feudenheimstr_aus
Station: mannheim_feudenheimstr_ein
Station: mannheim_konrad_adenauer
Station: mannheim_kurpfalz
Station: mannheim_luzenberg
Station: mannheim_renz
Station: mannheim_schlosspark
Station: mannheim_schwetzinger
Station: ravensburg_bahnhof
Station: ravensburg_eishalle
Station: ravensburg_meer_ab
Station: ravensburg_meer_auf
Station: stuttgart_boeblinger
Station: stuttgart_insel
Station: stuttgart_kirchheimer
Station: stuttgart_koenig_karls
Station: stuttgart_kraeherwald
Station: stuttgart_kremmler
Stati

In [26]:
# add informative columns to each dataframe
for alias in STATIONS:
    df = dfs[alias]
    city = alias.split("_")[0]
    
    # add public holiday column
    df['public_holiday'] = False
    for date in public_holidays:
        holiday_date = pd.to_datetime(date)
        df.loc[df['datetime'].dt.date == holiday_date.date(), 'public_holiday'] = True
    
    # add school holiday column
    df['school_holiday'] = False
    for start, end in school_holidays:
        start_date = pd.to_datetime(start)
        end_date = pd.to_datetime(end)
        df.loc[(df['datetime'] >= start_date) & (df['datetime'] <= end_date), 'school_holiday'] = True
    
    # add lecture free period column if city has university
    df['lecture_free'] = False
    if city in lecture_free_periods:
        for start, end in lecture_free_periods[city]:
            start_date = pd.to_datetime(start)
            end_date = pd.to_datetime(end)
            df.loc[(df['datetime'] >= start_date) & (df['datetime'] <= end_date), 'lecture_free'] = True
    
    # # if public holiday is True, set school_holiday and lecture_free to True
    # df.loc[df['public_holiday'] == True, 'school_holiday'] = True
    # df.loc[df['public_holiday'] == True, 'lecture_free'] = True
    
    
    # add weekday and hour columns
    df['weekday'] = df['datetime'].dt.day_name()
    df['hour'] = df['datetime'].dt.hour
    
    # add is_weekend column
    df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday'])
    
    # add workday column
    df['is_workday'] = ~df['is_weekend'] & ~df['public_holiday']
    
    dfs[alias] = df
    


In [27]:
# add rain related columns
for alias in STATIONS:
    df = dfs[alias]
    # add is_raining column
    df['is_raining'] = df['rain'] > 0.0
    # add rain_intensity column
    def rain_intensity(rain_mm):
        if rain_mm > 0.0 and rain_mm < 0.5:
            return 'light_drizzle'
        elif rain_mm >= 0.5 and rain_mm < 1.0:
            return 'strong_drizzle'
        elif rain_mm >= 1.0 and rain_mm < 2.0:
            return 'light_rain'
        elif rain_mm >= 2.0 and rain_mm < 5.0:
            return 'moderate_rain'
        elif rain_mm >= 5.0:
            return 'heavy_rain'
    df['rain_intensity'] = df['rain'].apply(rain_intensity)
    
    # add rain_last_hour column
    df['rain_last_hour'] = df['rain'].shift(1).fillna(0.0) > 0.0
    # add rain_last_2_hours column
    df['rain_last_2_hours'] = df['rain'].shift(2).fillna(0.0) > 0.0
    # add rain_last_3_hours column
    df['rain_last_3_hours'] = df['rain'].shift(3).fillna(0.0) > 0.0
    
    # add rain_next_hour column
    df['rain_next_hour'] = df['rain'].shift(-1).fillna(0.0) > 0.0
    # add rain_next_2_hours column
    df['rain_next_2_hours'] = df['rain'].shift(-2).fillna(0.0) > 0.0
    # add rain_next_3_hours column
    df['rain_next_3_hours'] = df['rain'].shift(-3).fillna(0.0) > 0.0
    
    # add rain during day column (6am to 10pm)
    df['rain_during_day'] = False
    df.loc[(df['datetime'].dt.hour >= 6) & (df['datetime'].dt.hour <= 22) & (df['rain'] > 0.0), 'rain_during_day'] = True
    
    dfs[alias] = df

In [28]:
# perform a regression analysis to investigate the influence of weather and holidays on bike counts
import statsmodels.api as sm
import numpy as np

for alias in STATIONS:
    df = dfs[alias].copy()
    
    # Drop rows with missing bike counts
    df = df.dropna(subset=['bike'])
    
    # Define independent variables
    X = df[['temp', 'rain', 'is_raining', 'rain_intensity', 'rain_last_hour', 'rain_last_2_hours', 'rain_last_3_hours', 'rain_next_hour', 'rain_next_2_hours', 'rain_next_3_hours', 'rain_during_day', 'wind', 'humidity', 'clouds', 'public_holiday', 'lecture_free', 'weekday', 'is_workday', 'is_weekend', 'hour']]
    
    # Convert categorical variables to numeric (one-hot encoding for weekday)
    X = pd.get_dummies(X, columns=['weekday'], drop_first=True)
    X = pd.get_dummies(X, columns=['rain_intensity'], drop_first=True)
    
    # Convert boolean columns to numeric
    X = X.astype(int)
    
    # Add a constant term for the intercept
    X = sm.add_constant(X)
    
    # Define dependent variable
    y = df['bike']
    
    # Fit the regression model
    model = sm.OLS(y, X).fit()
    
    # Print the summary of the regression results
    print(f"Regression results for station: {alias}")
    print(model.summary())
    print("\n\n")

Regression results for station: freiburg_eschholz
                            OLS Regression Results                            
Dep. Variable:                   bike   R-squared:                       0.446
Model:                            OLS   Adj. R-squared:                  0.445
Method:                 Least Squares   F-statistic:                     759.0
Date:                Fri, 23 Jan 2026   Prob (F-statistic):               0.00
Time:                        12:53:48   Log-Likelihood:            -1.4660e+05
No. Observations:               25507   AIC:                         2.933e+05
Df Residuals:                   25479   BIC:                         2.935e+05
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------

In [29]:
# # now perform a negative binomial regression analysis
# import statsmodels.api as sm
# import numpy as np

# for alias in STATIONS:
#     df = dfs[alias].copy()
    
#     # Drop rows with missing bike counts
#     df = df.dropna(subset=['bike'])
    
#     # Define independent variables
#     X = df[['temp', 'rain', 'is_raining', 'rain_intensity', 'rain_last_hour', 'rain_last_2_hours', 'rain_last_3_hours', 'rain_next_hour', 'rain_next_2_hours', 'rain_next_3_hours', 'rain_during_day', 'wind', 'humidity', 'clouds', 'public_holiday', 'lecture_free', 'weekday', 'is_workday', 'is_weekend', 'hour']]
    
#     # Convert categorical variables to numeric (one-hot encoding for weekday)
#     X = pd.get_dummies(X, columns=['weekday'], drop_first=True)
#     X = pd.get_dummies(X, columns=['rain_intensity'], drop_first=True)
    
#     # Convert boolean columns to numeric
#     X = X.astype(int)
    
#     # Add a constant term for the intercept
#     X = sm.add_constant(X)
    
#     # Define dependent variable
#     y = df['bike']
    
#     # Fit the negative binomial regression model
#     model = sm.GLM(y, X, family=sm.families.NegativeBinomial()).fit()
    
#     # Print the summary of the regression results
#     print(f"Negative Binomial Regression results for station: {alias}")
#     print(model.summary())
#     print("\n\n")

In [30]:
# now perform a log-linear regression analysis
import statsmodels.api as sm
import numpy as np

for alias in STATIONS:
    df = dfs[alias].copy()
    
    # Drop rows with missing bike counts
    df = df.dropna(subset=['bike'])
    
    # Define independent variables
    X = df[['temp', 'rain', 'is_raining', 'rain_intensity', 'rain_last_hour', 'rain_last_2_hours', 'rain_last_3_hours', 'rain_next_hour', 'rain_next_2_hours', 'rain_next_3_hours', 'rain_during_day', 'wind', 'humidity', 'clouds', 'public_holiday', 'lecture_free', 'weekday', 'is_workday', 'is_weekend', 'hour']]
    
    # Convert categorical variables to numeric (one-hot encoding for weekday)
    X = pd.get_dummies(X, columns=['weekday'], drop_first=True)
    X = pd.get_dummies(X, columns=['rain_intensity'], drop_first=True)
    
    # Convert boolean columns to numeric
    X = X.astype(int)
    
    # Add a constant term for the intercept
    X = sm.add_constant(X)
    
    # Define dependent variable (log-transform bike counts)
    y = np.log(df['bike'] + 1)  # add 1 to avoid log(0)
    
    # Fit the log-linear regression model
    model = sm.OLS(y, X).fit()
    
    # Print the summary of the regression results
    print(f"Log-Linear Regression results for station: {alias}")
    print(model.summary())
    print("\n\n")

Log-Linear Regression results for station: freiburg_eschholz
                            OLS Regression Results                            
Dep. Variable:                   bike   R-squared:                       0.507
Model:                            OLS   Adj. R-squared:                  0.507
Method:                 Least Squares   F-statistic:                     972.4
Date:                Fri, 23 Jan 2026   Prob (F-statistic):               0.00
Time:                        12:53:51   Log-Likelihood:                -31133.
No. Observations:               25507   AIC:                         6.232e+04
Df Residuals:                   25479   BIC:                         6.255e+04
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------

In [31]:
# same but with less independent variables to avoid multicollinearity
import statsmodels.api as sm
import numpy as np  

for alias in STATIONS:
    df = dfs[alias].copy()
    
    # Drop rows with missing bike counts
    df = df.dropna(subset=['bike'])
    
    # Define independent variables
    X = df[['temp', 'rain_intensity', 'school_holiday', 'is_workday', 'hour']]
    
    # Convert categorical variables to numeric (one-hot encoding for weekday)
    X = pd.get_dummies(X, columns=['rain_intensity'], drop_first=True)
    
    # Convert boolean columns to numeric
    X = X.astype(int)
    
    # Add a constant term for the intercept
    X = sm.add_constant(X)
    
    # Define dependent variable (log-transform bike counts)
    y = np.log(df['bike'] + 1)  # add 1 to avoid log(0)
    
    # Fit the log-linear regression model
    model = sm.OLS(y, X).fit()
    
    # Print the summary of the regression results
    print(f"Log-Linear Regression (reduced) results for station: {alias}")
    print(model.summary())
    print("\n\n")

Log-Linear Regression (reduced) results for station: freiburg_eschholz
                            OLS Regression Results                            
Dep. Variable:                   bike   R-squared:                       0.444
Model:                            OLS   Adj. R-squared:                  0.444
Method:                 Least Squares   F-statistic:                     2547.
Date:                Fri, 23 Jan 2026   Prob (F-statistic):               0.00
Time:                        12:53:53   Log-Likelihood:                -32676.
No. Observations:               25507   AIC:                         6.537e+04
Df Residuals:                   25498   BIC:                         6.544e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------